# 📊 Notebook 1: Exploratory Data Analysis (EDA)

**Môn học:** Học Máy (CO3001) — HK I (2025-2026)  
**Nhóm:** Hoàng Xuân Bách · Nguyễn Việt Hùng · Trần Văn Hùng  

---
### Mục tiêu
- Hiểu cấu trúc và đặc điểm của dữ liệu
- Phát hiện missing values và outliers
- Phân tích phân phối và tương quan giữa các features
- Visualize dữ liệu để có cái nhìn trực quan

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

# Import custom modules
import sys
sys.path.append('..')
from modules import config
from modules.preprocessing import check_missing_values

print('✅ Libraries loaded successfully!')

## 2. Load Data

In [ ]:
# TODO: Thay đường dẫn bằng file dataset của bạn
# df = pd.read_csv('../data/raw/your_dataset.csv')

# ── Ví dụ tạo data giả để test pipeline ──
from sklearn.datasets import make_classification
X_demo, y_demo = make_classification(n_samples=1000, n_features=15,
                                     n_informative=10, n_redundant=3,
                                     random_state=42)
col_names = [f'feature_{i}' for i in range(15)]
df = pd.DataFrame(X_demo, columns=col_names)
df['target'] = y_demo

# Thêm missing values và categorical để demo
import numpy as np
np.random.seed(42)
df.loc[np.random.choice(df.index, 80), 'feature_0'] = np.nan
df.loc[np.random.choice(df.index, 50), 'feature_1'] = np.nan
df['category_col'] = np.random.choice(['A','B','C'], size=len(df))

print(f'Dataset shape: {df.shape}')
df.head()

## 3. Thông Tin Cơ Bản

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Basic Statistics ===')
df.describe().round(2)

In [ ]:
print('Data types summary:')
print(df.dtypes.value_counts())
print(f'\nTotal samples : {len(df):,}')
print(f'Total features: {df.shape[1] - 1}')
print(f'Duplicates    : {df.duplicated().sum()}')

## 4. Phân Tích Missing Values

In [ ]:
missing_df = check_missing_values(df)

if missing_df.empty:
    print('✅ Không có missing values!')
else:
    print(f'⚠️  Có {len(missing_df)} cột có missing values:\n')
    print(missing_df)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 5))
    missing_df['Missing_Percentage'].plot(kind='bar', ax=ax, color='salmon', edgecolor='black')
    ax.set_title('Missing Values (%) by Feature', fontsize=14, fontweight='bold')
    ax.set_ylabel('Percentage (%)')
    ax.set_xlabel('Feature')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 5. Phân Tích Biến Mục Tiêu (Target)

In [ ]:
target_col = 'target'  # TODO: Thay bằng tên cột target của bạn

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
value_counts = df[target_col].value_counts()
axes[0].bar(value_counts.index.astype(str), value_counts.values,
            color=sns.color_palette('pastel'), edgecolor='black')
axes[0].set_title('Target Distribution (Count)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
for i, v in enumerate(value_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(value_counts.values, labels=value_counts.index.astype(str),
            autopct='%1.1f%%', colors=sns.color_palette('pastel'),
            startangle=90)
axes[1].set_title('Target Distribution (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Class distribution:')
print(df[target_col].value_counts())
print('\nClass percentage:')
print((df[target_col].value_counts(normalize=True)*100).round(2))

## 6. Phân Tích Numeric Features

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(target_col).tolist()
print(f'Numeric features ({len(numeric_cols)}): {numeric_cols}')

In [ ]:
# Distribution plots
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten() if n_rows > 1 else axes

for idx, col in enumerate(numeric_cols):
    ax = axes[idx] if isinstance(axes, np.ndarray) else axes
    df[col].hist(bins=30, ax=ax, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'{col}', fontsize=11)
    ax.set_xlabel('')

for idx in range(len(numeric_cols), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Distribution of Numeric Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Box plots – phát hiện outliers
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten() if n_rows > 1 else axes

for idx, col in enumerate(numeric_cols):
    ax = axes[idx] if isinstance(axes, np.ndarray) else axes
    df.boxplot(column=col, ax=ax, vert=True,
               patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax.set_title(f'{col}', fontsize=11)

for idx in range(len(numeric_cols), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Box Plots – Outlier Detection', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Phân Tích Categorical Features

In [ ]:
categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
print(f'Categorical features ({len(categorical_cols)}): {categorical_cols}')

for col in categorical_cols:
    print(f'\n{col}: {df[col].nunique()} unique values')
    print(df[col].value_counts())

In [ ]:
if categorical_cols:
    fig, axes = plt.subplots(1, len(categorical_cols), figsize=(7*len(categorical_cols), 5))
    if len(categorical_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, categorical_cols):
        vc = df[col].value_counts()
        ax.bar(vc.index, vc.values, color=sns.color_palette('Set2'), edgecolor='black')
        ax.set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
        ax.set_xlabel(col)
        ax.set_ylabel('Count')
    plt.tight_layout()
    plt.savefig('../reports/figures/categorical_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Phân Tích Tương Quan (Correlation)

In [ ]:
corr_matrix = df[numeric_cols + [target_col]].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, mask=mask,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Các cặp tương quan cao (|corr| > 0.8)
threshold = 0.8
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > threshold:
            high_corr.append((corr_matrix.columns[i],
                              corr_matrix.columns[j],
                              corr_matrix.iloc[i, j]))

if high_corr:
    print(f'⚠️  Các cặp feature có |correlation| > {threshold}:')
    for f1, f2, c in high_corr:
        print(f'  {f1} ↔ {f2}: {c:.3f}')
else:
    print('✅ Không có cặp feature nào có tương quan quá cao.')

## 9. Feature vs Target

In [ ]:
# Phân phối của mỗi numeric feature theo từng class
show_cols = numeric_cols[:6]  # hiển thị tối đa 6 features

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

palette = sns.color_palette('Set1', n_colors=df[target_col].nunique())

for idx, col in enumerate(show_cols):
    for i, cls in enumerate(sorted(df[target_col].unique())):
        axes[idx].hist(df[df[target_col]==cls][col].dropna(),
                       bins=25, alpha=0.6, label=f'Class {cls}',
                       color=palette[i], edgecolor='none')
    axes[idx].set_title(f'{col} by Target', fontsize=11, fontweight='bold')
    axes[idx].legend()

for idx in range(len(show_cols), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Feature Distribution by Target Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/features_by_target.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Tổng Kết EDA

*(Điền các nhận xét của nhóm vào đây)*

| Hạng mục | Kết quả |
|:---------|:--------|
| Số mẫu | ??? |
| Số features | ??? |
| Missing values | ??? cột |
| Imbalanced? | ??? |
| Outliers | ??? |
| High correlation pairs | ??? |

**Các vấn đề cần xử lý trong bước preprocessing:**
1. ...
2. ...
3. ...

In [ ]:
print('=== EDA Summary ===')
print(f'Samples          : {len(df):,}')
print(f'Numeric features : {len(numeric_cols)}')
print(f'Categorical feat.: {len(categorical_cols)}')
print(f'Missing col count: {df.isnull().any().sum()}')
print(f'Duplicate rows   : {df.duplicated().sum()}')
print(f'High-corr pairs  : {len(high_corr)}')
print('\n✅ EDA hoàn thành! Chạy notebook 02_Preprocessing tiếp theo.')